# ClusterSourceSelect stage functionalities

This notebook aims at presenting the `ClusterSourceSelect` stage of the TXpipe clusters extension. This stage selects the background galaxies of each cluster of a cluster catalog. The shear-related quantities of those galaxies (tangential and cross shear components, weights) are computed by the `ClusterEnsembleProfiles` stage, which saves them for each galaxy

*This notebook was developed by Céline COmbet and Camille Avestruz for the DESC CL_Cosmo_Pipeline team.*
___

In [1]:
import os
from pprint import pprint

import ceci
import h5py

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import Image

Make sure to change your path in the next cell that leads to your TXPipe directory. See examples for IN2P3 and NERSC below.

In [ ]:
# user specific paths -- IN2P3 example
my_txpipe_dir = "/pbs/home/m/mricci/throng_mricci/desc/TXPipe"
# my_txpipe_dir = "/pbs/throng/lsst/users/ccombet/TXPipe"

# user specific paths -- NERSC example
# my_txpipe_dir = "/pscratch/sd/a/avestruz/TXPipe"

os.chdir(my_txpipe_dir)

import txpipe
import txpipe.extensions

# 1 deg$^2$ sample running directly in Jupyter

First we will do some runs on the 1 deg^2 example data set with around 80k galaxies. This is small enough that we can do it all in jupyter.

The data set, which is based on CosmoDC2, contains pre-computed photo-z and and contains a RedMapper cluster catalog for the field.

## This initiates and run the stage

In [3]:
print("Options for this pipeline and their defaults:")
print(txpipe.extensions.CLClusterSourceSelect.config_options)

Options for this pipeline and their defaults:
{'chunk_rows': 100000, 'max_radius': 10.0, 'delta_z': 0.1, 'redshift_cut_criterion': 'zmode', 'redshift_weight_criterion': 'zmode', 'redshift_cut_criterion_pdf_fraction': 0.9, 'subtract_mean_shear': False, 'coordinate_system': 'celestial', 'use_true_shear': False}


In [ ]:
pipe_stage = txpipe.extensions.CLClusterSourceSelect.make_stage(
    # catalogs
    shear_catalog="data/example/inputs/metadetect_shear_catalog.hdf5",
    cluster_catalog="./data/example/inputs/cluster_catalog.hdf5",
    source_photoz_pdfs="data/example/inputs/photoz_pdfs.hdf5",
    # The source selection (shear_selection) and the tomographic bins (shear_tomography_catalog)
    # were made by previously running the metadetect test pipeline, examples/test/metadetect
    shear_selection="data/test/metadetect/outputs_metadetect/shear_selection.hdf5",
    shear_tomography_catalog="data/test/metadetect/outputs_metadetect/shear_tomography_catalog.hdf5",
    # Fiducial cosmology: it is needed to get physical distances as we are
    # currently selecting sources based on projected distance (in Mpc)
    # from cluster center
    fiducial_cosmology="./data/fiducial_cosmology.yml",
    # This is the output for this stage
    cluster_shear_selection="data/mocks/cosmodc2/outputs-1deg2-CL/cluster_shear_selection.hdf5",
    # This contains all the options for this stage. Default config options will be updated
    config="./examples/mocks/cosmodc2/Cluster_pipelines/config-1deg2-CL.yml",
)

In [ ]:
# Check the new config options
pipe_stage.config

In [ ]:
pipe_stage.run()
pipe_stage.finalize()

## Checking out the output

To avoid making lots and lots of copies of the data, this stage has not made a catalog, but instead made an index into the other catalogs: for each cluster, the index of its sources in the shear catalog and their distance to the cluster.

The shear-related quantities are computed by the ensemble stage, which saves them for the sources of each cluster in its bins. We have a helper class which is designed to match up the cluster catalog, these sources and the profiles, and collect the results for each cluster.

In [ ]:
# The shears, weights and profiles are computed by the ensemble stage, which needs
# the clusters in bins. It saves the sources of each cluster with these quantities
# in cluster_source_quantities, which the helper class below reads.
step_bin = txpipe.extensions.CLClusterBinningRedshiftRichness.make_stage(
    cluster_catalog="./data/example/inputs/cluster_catalog.hdf5",
    cluster_catalog_tomography="data/mocks/cosmodc2/outputs-1deg2-CL/cluster_catalog_tomography.hdf5",
    richedge=[1, 80],
    zedge=[0.2, 1.0],
)
step_bin.run()
step_bin.finalize()

step_ens = txpipe.extensions.CLClusterEnsembleProfiles.make_stage(
    shear_catalog="data/example/inputs/metadetect_shear_catalog.hdf5",
    shear_tomography_catalog="data/test/metadetect/outputs_metadetect/shear_tomography_catalog.hdf5",
    fiducial_cosmology="./data/fiducial_cosmology.yml",
    cluster_shear_selection="data/mocks/cosmodc2/outputs-1deg2-CL/cluster_shear_selection.hdf5",
    cluster_catalog_tomography="data/mocks/cosmodc2/outputs-1deg2-CL/cluster_catalog_tomography.hdf5",
    source_photoz_pdfs="data/example/inputs/photoz_pdfs.hdf5",
    cluster_profiles="data/mocks/cosmodc2/outputs-1deg2-CL/cluster_profiles.pkl",
    cluster_source_quantities="data/mocks/cosmodc2/outputs-1deg2-CL/cluster_source_quantities.hdf5",
    config="./examples/mocks/cosmodc2/Cluster_pipelines/config-1deg2-CL.yml",
)
step_ens.run()
step_ens.finalize()

In [ ]:
ccc = txpipe.extensions.CombinedClusterCatalog(
    cluster_catalog="./data/example/inputs/cluster_catalog.hdf5",
    cluster_source_quantities="data/mocks/cosmodc2/outputs-1deg2-CL/cluster_source_quantities.hdf5",
    cluster_profiles="data/mocks/cosmodc2/outputs-1deg2-CL/cluster_profiles.pkl",
    shear_catalog="./data/example/inputs/metadetect_shear_catalog.hdf5",
)

In [ ]:
print(f"We have {ccc.ncluster} clusters")

We can extract the cluster catalog info by index (0 to `ccc.ncluster - 1`, the clusters in the bins of the ensemble stage):

In [ ]:
ind = 0
cluster_info = ccc.get_cluster_info(ind)
cluster_info

And also the shear catalog associated with that cluster, again by index, in the CLMM data format

In [ ]:
bg_cat = ccc.get_background_shear_catalog(ind)
bg_cat[0:3]

In [ ]:
bg_cat = ccc.get_background_shear_catalog(ind)
bg_cat[0:3]

In [ ]:
len(bg_cat["source_index"])

In [ ]:
bg_cat.meta

In [13]:
import clmm
clmm_cosmo = clmm.cosmology.ccl.CCLCosmology()
clmm_cosmo.set_be_cosmo()

In [ ]:
prof_Mpc = clmm.make_radial_profile( [bg_cat["tangential_comp"], bg_cat["cross_comp"]], bg_cat["distance_arcmin"],
                     "arcmin", "Mpc", bins=6, cosmo=clmm_cosmo , z_lens=cluster_info['redshift'], weights= bg_cat["weight"])

In [ ]:
plt.semilogx(prof_Mpc["radius"], prof_Mpc["p_0"], "o-")
plt.semilogx(prof_Mpc["radius"], prof_Mpc["p_1"], "o-")

In [ ]:
plt.semilogx(prof_Mpc["radius"], prof_Mpc["p_0"], "o-")
plt.semilogx(prof_Mpc["radius"], prof_Mpc["p_1"], "o-")

# 20 deg$^2$ example using the pipeline approach

In [ ]:
# Read the appropriate pipeline configuration, and ask for a flow-chart.
pipeline_file = "examples/mocks/cosmodc2/Cluster_pipelines/pipeline-20deg2-CL-in2p3.yml"
flowchart_file = "CLClusterShearCat.png"

pipeline_config = ceci.Pipeline.build_config(
    pipeline_file, flow_chart=flowchart_file, dry_run=True
)

# Run the flow-chart pipeline
ceci.run_pipeline(pipeline_config);

In [ ]:
Image(flowchart_file)

In [ ]:
ccc = txpipe.extensions.CombinedClusterCatalog.from_pipeline_file(pipeline_file)

In [8]:
# Get the path to the data files
data_dir = "data/cosmodc2/20deg2"

# open the cluster and shear catalogs
clusters = h5py.File(os.path.join(data_dir, "cluster_catalog.hdf5"), "r")
shear = h5py.File(os.path.join(data_dir, "shear_catalog.hdf5"), "r")

In [10]:
array([4345., 4252., 4136., 6091., 7122.])

NameError: name 'array' is not defined

In [ ]:
print(f"We have {ccc.ncluster} clusters")

In [12]:
cluster_z = clusters['clusters/redshift'][:]
cluster_richness = clusters['clusters/richness'][:]
cluster_id = clusters['clusters/cluster_id'][:]

In [ ]:
# index of the cluster in ccc
ind = int(np.flatnonzero(ccc.cluster_ids == 4252)[0])
ind

In [ ]:
cluster_info = ccc.get_cluster_info(ind)
cluster_info

In [ ]:
bg_cat = ccc.get_background_shear_catalog(ind)
bg_cat[0:3]

In [ ]:
np.max(bg_cat["distance_arcmin"])

In [ ]:
len(bg_cat)

In [ ]:
gal_ra = bg_cat["ra"]
gal_dec = bg_cat["dec"]

In [ ]:
gal_ra.size

In [ ]:
plt.scatter(gal_ra, gal_dec, c = bg_cat["distance_arcmin"])
plt.colorbar()

In [18]:
import clmm

In [19]:
clmm_cosmo = clmm.cosmology.ccl.CCLCosmology()
clmm_cosmo.set_be_cosmo()

In [20]:
dist_Mpc = clmm.utils.convert_units(bg_cat["distance_arcmin"],'arcmin', 'Mpc', 
                         cluster_info['redshift'], clmm_cosmo)

In [ ]:
plt.scatter(gal_ra, gal_dec, c = dist_Mpc)
plt.colorbar()

In [ ]:
bg_cat

In [23]:
prof_Mpc = clmm.make_radial_profile( [bg_cat["tangential_comp"], bg_cat["cross_comp"]], bg_cat["distance_arcmin"],
                     "arcmin", "Mpc", cosmo=clmm_cosmo , z_lens=cluster_info['redshift'], weights= bg_cat["weight"])

In [ ]:
prof_Mpc

In [ ]:
plt.semilogx(prof_Mpc["radius"], prof_Mpc["p_0"], "o-")
plt.semilogx(prof_Mpc["radius"], prof_Mpc["p_1"], "o-")

In [ ]:
radial_bins = clmm.dataops.make_bins(0.2, 5., nbins=10, method="evenlog10width")
radial_bins

In [28]:
prof_rad = clmm.make_radial_profile( [bg_cat["tangential_comp"], bg_cat["cross_comp"]], bg_cat["distance_arcmin"],
                     "arcmin", "Mpc", bins=radial_bins, cosmo=clmm_cosmo , z_lens=cluster_info['redshift'], weights= bg_cat["weight"])

In [ ]:
plt.semilogx(prof_rad["radius"], prof_rad["p_0"], "o-")
plt.semilogx(prof_rad["radius"], prof_rad["p_1"], "o-")

In [ ]:
plt.semilogx(prof_rad["radius"], prof_rad["weights_sum"], "o-")

In [ ]:
plt.semilogx(prof_rad["radius"], prof_rad["n_src"], "o-")
plt.ylim(0,1000)

In [ ]:
plt.scatter(bg_cat['true_g1'], bg_cat['g1'])
plt.xlabel('true_g1')
plt.ylabel('g1')

In [ ]:
plt.scatter(bg_cat['true_g2'], bg_cat['g2'])
plt.xlabel('true_g2')
plt.ylabel('g2')

In [ ]:
radial_bins = clmm.dataops.make_bins(0.2, 5., nbins=10, method="evenlog10width")
radial_bins

In [ ]:

galcat = clmm.GCData(bg_cat, meta=bg_cat.meta)
galcat['theta'] = galcat['distance_arcmin']*np.pi/(60*180) # CLMM galcat requires a column called "theta" in radians
galcat['z'] = np.zeros(len(galcat)) # clmm needs a column named 'z' but all computation have been done                    
                
gc_object = clmm.GalaxyCluster(int(cluster_info['cluster_id']), cluster_info['ra'], cluster_info['dec'],
                               cluster_info['redshift'], galcat )

In [ ]:
gc_object.make_radial_profile(
                    "Mpc", 
                    bins=radial_bins,
                    cosmo=clmm_cosmo, 
                    tan_component_in = "tangential_comp", # name given in the CLClusterEnsembleProfiles stage
                    cross_component_in = "cross_comp", # name given in the CLClusterEnsembleProfiles stage
                    tan_component_out = "tangential_comp2",
                    cross_component_out = "cross_comp2",
                    use_weights=True,
                    weights_in = "weight", # name given in the CLClusterEnsembleProfiles stage
                    weights_out = "W_l",
                    include_empty_bins = True, 
                       )


In [ ]:
gc_object.plot_profiles('tangential_comp2', 'tangential_comp2_err', 'cross_comp2', 'cross_comp2_err', xscale='log')

In [ ]:
gc_object.plot_profiles('tangential_comp2', 'tangential_comp2_err', 'cross_comp2', 'cross_comp2_err', xscale='log')

In [ ]:
gc_object.plot_profiles('tangential_comp2', 'tangential_comp2_err', 'cross_comp2', 'cross_comp2_err', xscale='log')

In [37]:
cluster_ensemble = clmm.ClusterEnsemble(0)

In [ ]:
cluster_ensemble.add_individual_radial_profile(
                    galaxycluster=gc_object,
                    profile_table=gc_object.profile, 
                    tan_component="tangential_comp2",
                    cross_component="cross_comp2",
                    weights="W_l")

In [ ]:
gc_object.make_radial_profile(
                    "Mpc", 
                    bins=radial_bins,
                    cosmo=clmm_cosmo, 
                    tan_component_in = "tangential_comp", # name given in the CLClusterEnsembleProfiles stage
                    cross_component_in = "cross_comp", # name given in the CLClusterEnsembleProfiles stage
                    tan_component_out = "tangential_comp",
                    cross_component_out = "cross_comp",
                    use_weights=True,
                    weights_in = "weight", # name given in the CLClusterEnsembleProfiles stage
                    weights_out = "W_l",
                    include_empty_bins = True 
                    )


In [ ]:
comp = clmm.dataops.compute_tangential_and_cross_components(
    cluster_info["ra"],
    cluster_info["dec"],
    bg_cat["ra"],
    bg_cat["dec"],
    bg_cat["g1"],
    bg_cat["g2"],
    coordinate_system="euclidean",
)

In [ ]:
comp = clmm.dataops.compute_tangential_and_cross_components(
    cluster_info["ra"],
    cluster_info["dec"],
    bg_cat["ra"],
    bg_cat["dec"],
    bg_cat["g1"],
    bg_cat["g2"],
    coordinate_system="celestial",
)

In [ ]:
prof = clmm.make_radial_profile(comp, bg_cat["distance_arcmin"], "arcmin", "arcmin")

In [ ]:
prof

In [ ]:
plt.semilogx(prof["radius"], prof["p_1"], "o-")
plt.semilogx(prof["radius"], prof["p_2"], "o-")

In [ ]:
plt.semilogx(prof["radius"], prof["p_1"], "o-")
plt.semilogx(prof["radius"], prof["p_2"], "o-")

In [ ]:
clmm.dataops.compute_tangential_and_cross_components(
    cluster_info["ra"],
    cluster_info["dec"],
    bg_cat["ra"],
    bg_cat["dec"],
    bg_cat["g1"],
    bg_cat["g2"],
    coordinate_system="celestial",
)

In [ ]:
plt.scatter(bg_cat["g1"], bg_cat["g2"], marker=".")

In [ ]:
plt.scatter(bg_cat["distance_arcmin"], bg_cat["tangential_comp"])

In [ ]:
plt.scatter(bg_cat["distance_arcmin"], bg_cat["cross_comp"])